In [ ]:

from google.colab import drive

drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
from pathlib import Path

path = Path('/content/drive/MyDrive/03_Proiecte/LLM_Scratch')

input_file = path / 'input.txt'
more_file = path / 'more.txt'

print(path.exists())
print(input_file.exists())
print(more_file.exists())



True
True
True


In [ ]:
with open (input_file, 'r' , encoding= 'utf-8') as file:
  text=file.read()

print(text[:100])
print("---" *50)

chars= sorted(list(set(text)))
vocab_size = len(chars)
print(f'Numer of characters: {len(chars)}')
print(chars)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
------------------------------------------------------------------------------------------------------------------------------------------------------
Numer of characters: 65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


Neuronal Network can t process characters so we need to transform those charcters in TOKENS ( numbers )


---

stoi = String to Integer (caracter → ID).


itos = Integer to String (ID → caracter).

In [ ]:
stoi = {}
for nr_id,c in enumerate(chars):
  stoi[c]= nr_id

itos = {}
for nr_id,c in enumerate(chars):
  itos[nr_id]=c


print("STOI")
for c, nr_id in stoi.items():
    print(repr(c), "->", nr_id)

print("\nITOS")
for nr_id, c in itos.items():
    print(nr_id, "->", repr(c))


# we use repr(c) to represent special charcters line \n - new line

STOI
'\n' -> 0
' ' -> 1
'!' -> 2
'$' -> 3
'&' -> 4
"'" -> 5
',' -> 6
'-' -> 7
'.' -> 8
'3' -> 9
':' -> 10
';' -> 11
'?' -> 12
'A' -> 13
'B' -> 14
'C' -> 15
'D' -> 16
'E' -> 17
'F' -> 18
'G' -> 19
'H' -> 20
'I' -> 21
'J' -> 22
'K' -> 23
'L' -> 24
'M' -> 25
'N' -> 26
'O' -> 27
'P' -> 28
'Q' -> 29
'R' -> 30
'S' -> 31
'T' -> 32
'U' -> 33
'V' -> 34
'W' -> 35
'X' -> 36
'Y' -> 37
'Z' -> 38
'a' -> 39
'b' -> 40
'c' -> 41
'd' -> 42
'e' -> 43
'f' -> 44
'g' -> 45
'h' -> 46
'i' -> 47
'j' -> 48
'k' -> 49
'l' -> 50
'm' -> 51
'n' -> 52
'o' -> 53
'p' -> 54
'q' -> 55
'r' -> 56
's' -> 57
't' -> 58
'u' -> 59
'v' -> 60
'w' -> 61
'x' -> 62
'y' -> 63
'z' -> 64

ITOS
0 -> '\n'
1 -> ' '
2 -> '!'
3 -> '$'
4 -> '&'
5 -> "'"
6 -> ','
7 -> '-'
8 -> '.'
9 -> '3'
10 -> ':'
11 -> ';'
12 -> '?'
13 -> 'A'
14 -> 'B'
15 -> 'C'
16 -> 'D'
17 -> 'E'
18 -> 'F'
19 -> 'G'
20 -> 'H'
21 -> 'I'
22 -> 'J'
23 -> 'K'
24 -> 'L'
25 -> 'M'
26 -> 'N'
27 -> 'O'
28 -> 'P'
29 -> 'Q'
30 -> 'R'
31 -> 'S'
32 -> 'T'
33 -> 'U'
34 -> 'V'
35 -> '

WE take every character from the text and search it in the dictionary to coresponde a numeric id

In [ ]:
def encode(txt):
    rez = []

    for c in txt:
        rez.append(stoi[c])

    return rez


def decode(lista):
    rez = ""

    for nr in lista:
        rez += itos[nr]

    return rez


In [ ]:
mesaj = "Hello"

codificat = encode(mesaj)

print(codificat)
print(decode(codificat))

[20, 43, 50, 50, 53]
Hello


Transform the Tensor Id into Tensor PyTorch

In [ ]:
import torch
torch.manual_seed(1337)
# we encode every character into an numeric ID
data=torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


Split the data for Train \ Test

In [ ]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print(len(train_data))
print(len(val_data))

1003854
111540


In [ ]:
batch_size=4 # number of sequences
block_size=8 #number of tokens

def get_batch(split):

  if split == 'train_data':
    data=train_data
  else:
    data = val_data

  #we take random pozitions so the model doesn t learn from patterns
  pozitii=torch.randint(
      0,len(data) - block_size, (batch_size,)
  )

  x= []
  y = []

  for pozitie in pozitii:
    x.append(data[pozitie : pozitie+block_size])
    y.append ( data[pozitie+1 : pozitie+block_size+1])

  x = torch.stack(x)
  y = torch.stack(y)

  return x, y


xb, yb = get_batch('train')

print("INPUT")
print(xb)

print("TARGET")
print(yb)

print("SHAPE")
print(xb.shape, yb.shape)

INPUT
tensor([[ 6,  1, 52, 53, 58,  1, 58, 47],
        [ 6,  1, 54, 50, 39, 52, 58, 43],
        [ 1, 58, 46, 47, 57,  1, 50, 47],
        [ 0, 32, 46, 43, 56, 43,  1, 42]])
TARGET
tensor([[ 1, 52, 53, 58,  1, 58, 47, 50],
        [ 1, 54, 50, 39, 52, 58, 43, 58],
        [58, 46, 47, 57,  1, 50, 47, 60],
        [32, 46, 43, 56, 43,  1, 42, 53]])
SHAPE
torch.Size([4, 8]) torch.Size([4, 8])


# Avem 32 de predictii, fiecare cu 65 de logits (scoruri brute).
# CrossEntropy calculeaza intern log_softmax pentru fiecare grup de 65 de scoruri.
# Apoi verifica scorul/probabilitatea tokenului corect din targets.
# Calculeaza loss-ul pentru fiecare predictie si returneaza media erorilor.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#lookup matrix
class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    #lookup matrix
    self.token_embedding=nn.Embedding(vocab_size,vocab_size)

  def forward(self,xb, targets=None):
    #if xb is one logit, logits has the scores for that logit
    logits=self.token_embedding(xb)

    if targets is None:
      loss=None
    else:
      #(B,T,C)=(4,8,65)
      B, T, C = logits.shape
      logits = logits.reshape(B * T, C)  # (4,8,65) -> (32,65)
      targets = targets.reshape(B * T)  # (4,8) -> (32,)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  # Dezactiveaza calculul gradientilor in timpul generarii.
  # Nu antrenam modelul, doar folosim parametrii deja invatati.
  @torch.no_grad()
  def generate(self, idx, max_new_tokens):

      # idx = tensorul cu ID-urile caracterelor existente (B, T).
      # max_new_tokens = cate caractere noi vrem sa generam.

      # Repetam procesul pentru fiecare token nou pe care vrem sa-l generam.
      for _ in range(max_new_tokens):

          # Trimitem contextul (idx) in model, adica apelam forward().
          # Primim logits pentru fiecare pozitie si ignoram loss-ul (_),
          # deoarece nu avem raspunsuri corecte (targets).
          logits, _ = self(idx)

          # Pastram doar cele C scoruri de la ultima pozitie.
          # : = toate secventele, -1 = ultima pozitie, : = toate scorurile.
          # Forma: (B, T, C) -> (B, C).
          logits = logits[:, -1, :]

          # Transformam scorurile brute (logits) in probabilitati.
          # dim=-1 inseamna ultima dimensiune, cea cu C tokeni posibili.
          # Probabilitatile de pe fiecare rand insumeaza 100%.
          probs = F.softmax(logits, dim=-1)

          # Alegem aleatoriu un token nou pentru fiecare secventa,
          # tinand cont de probabilitati (tokenii mai probabili au sanse mai mari).
          # num_samples=1 -> un singur token pentru fiecare rand.
          # Rezultat: (B, 1).
          idx_next = torch.multinomial(probs, num_samples=1)

          # Adaugam tokenul generat la finalul fiecarei secvente.
          # dim=1 este dimensiunea secventei de tokeni.
          # Forma: (B, T) + (B, 1) -> (B, T+1).
          idx = torch.cat((idx, idx_next), dim=1)

      # Returnam contextul initial impreuna cu toti tokenii generati.
      return idx


model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print('Logits shape:', logits.shape)
print('Loss initial:', loss.item())
print('Parametri:', sum(p.numel() for p in model.parameters()))


Logits shape: torch.Size([32, 65])
Loss initial: 4.717945098876953
Parametri: 4225


Softmax is a function that transform brute scores (logits ) in probabilities between 0 an 1 and they have to result in a total of 1

In [ ]:
#Example
import torch
from torch.nn import functional as F

scor=torch.tensor([2.0,0.5,1])
probabilities=F.softmax(scor)

print(probabilities)
print(sum(probabilities))
# From this example we can see that the logit with the highest score has the biggest probabiliti

tensor([0.6285, 0.1402, 0.2312])
tensor(1.0000)


/tmp/ipykernel_1671/455634357.py:6: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probabilities=F.softmax(scor)


#6. Training Loop — Backpropagation + Optimizer


*   zero_grad() șterge gradienții vechi.
*   backward() îi calculează pe cei noi.
*   step() modifică efectiv ponderile. Modelul este Bigram, nu GPT.**bold text**


In [ ]:
max_iters = 3000
learning_rate = 1e-2
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.train()
for pas in range(max_iters):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if pas % 300 == 0 or pas == max_iters - 1:
        print(pas, round(loss.item(), 4))

0 4.5128
300 2.963
600 2.5925
900 2.6288
1200 2.3019
1500 2.5227
1800 2.3277
2100 2.9424
2400 2.9396
2700 2.5489
2999 2.0481


In [ ]:
bigram = BigramLanguageModel(vocab_size)  # Cream modelul si-l mutam pe GPU/CPU

xb, yb = get_batch('train')
logits_train, loss_initial = bigram(xb, yb)  # Cu targets -> logits aplatizate + loss
logits_generate, _ = bigram(xb)              # Fara targets -> logits (B,T,V)

print('Logits la train:', logits_train.shape)
print('Logits la generate:', logits_generate.shape)
print('Loss initial:', round(loss_initial.item(), 4))
print('Parametri Bigram:', sum(p.numel() for p in bigram.parameters()))

# Exemplu simplu Softmax - observam ca suma e 1.0
scoruri = torch.tensor([[3.0, 2.0, 1.0]])
probabilitati = F.softmax(scoruri, dim=-1)
print('Softmax exemplu:', probabilitati)
print('Suma:', probabilitati.sum().item())

Logits la train: torch.Size([32, 65])
Logits la generate: torch.Size([4, 8, 65])
Loss initial: 4.725
Parametri Bigram: 4225
Softmax exemplu: tensor([[0.6652, 0.2447, 0.0900]])
Suma: 1.0


In [ ]:
@torch.no_grad()                            # Evaluarea nu are nevoie de gradiente.
def estimeaza_loss(model, split, nr_batch=5):
    era_train = model.training              # Retinem modul initial al modelului.
    model.eval()                            # Trecem pe modul de evaluare.

    total = 0.0                             # Aici adunam loss-urile.
    for _ in range(nr_batch):
        xb, yb = get_batch(split)           # Luam un batch din train sau val.
        _, loss = model(xb, yb)             # Evaluam eroarea pe acest batch.
        total += loss.item()                # Convertim loss-ul intr-un numar Python.

    if era_train:
        model.train()                       # Revenim la training daca era activ.

    return total / nr_batch                 # Media loss-ului

In [ ]:
# Un numar mic de pasi pentru invatare rapida; creste dupa ce intelegi codul.
max_bigram = 150
lr_bigram = 1e-2

opt_bigram = torch.optim.AdamW(bigram.parameters(), lr=lr_bigram)  # Optimizerul

pasi_bigram = []                        # Retinem pasii pentru grafic
loss_bigram_train = []                  # Istoricul erorii pe train
loss_bigram_val = []                    # Istoricul erorii pe validation

bigram.train()                          # Modul antrenare

for pas in range(max_bigram):
    xb, yb = get_batch('train')         # 1. Alegem un batch de TRAIN
    logits, loss = bigram(xb, yb)      # 2. Forward + Cross-Entropy

    opt_bigram.zero_grad(set_to_none=True) # 3. Resetam gradientele vechi
    loss.backward()                    # 4. Calculam gradientele noi
    opt_bigram.step()                   # 5. Actualizam valorile din embedding

    # Masuram periodic ambele pierderi, pentru a vedea cum evolueaza.
    if pas % max(1, max_bigram // 5) == 0 or pas == max_bigram - 1:
        tr = estimeaza_loss(bigram, 'train')
        va = estimeaza_loss(bigram, 'val')
        pasi_bigram.append(pas)
        loss_bigram_train.append(tr)
        loss_bigram_val.append(va)
        print('Bigram', pas, 'train', round(tr, 3), 'val', round(va, 3))

Bigram 0 train 2.52 val 2.487
Bigram 30 train 2.505 val 2.502
Bigram 60 train 2.441 val 2.387
Bigram 90 train 2.542 val 2.506
Bigram 120 train 2.536 val 2.421
Bigram 149 train 2.678 val 2.401


In [ ]:
bigram.eval()

# Incepem cu un singur token prezent in vocabular.
primul_id = stoi[text[0]]
context_bigram = torch.tensor([[primul_id]], dtype=torch.long)

# Generam 250 de caractere noi (sau mai putine daca testezi foarte rapid).
rez_bigram = bigram.generate(context_bigram, max_new_tokens=250)
text_bigram = decode(rez_bigram[0].tolist())

print('--- TEXT BIGRAM ---')
print(text_bigram)


--- TEXT BIGRAM ---
F?
fPTRUQxhink.zbe ois cershtolivezemathemat hu t he tharerat DRDsthe t thouredenoomend l ord wine o matind, ps piste t:
TQmound IO:
T ame by!

s bel IENCRUC.
me,
Tu he hes&DENShof andessoowahof ldoo weyo MGRUCDI s:
Mingoustin s thoaingowingve idowa.

